# V1-10: 대표 기업 표현의 후보 구성 비교

이번 평가는 약칭·한글/영문 표기·오타 때문에 사용자가 의도한 기업이 후보에서 사라지는 문제를 확인한다. 특히 `삼전`, `하닉`, `엘지`처럼 실제 검색에서 나올 수 있는 짧은 표현을 포함한다.

이번 notebook은 **최종 기업을 자동 선택하는 평가가 아니다.** 후보 생성 단계의 재현율과 후보 수를 비교한다. 최종 선택은 후보가 하나로 안전하게 확정되는 경우에만 수행하고, 나머지는 이후 사용자 확인 정책으로 처리한다.

## 1. 비교 대상과 정답 기준

대표적인 국내 상장기업의 정식 법인명·종목코드는 실행 시 내려받는 OpenDART 기업코드 목록에서 다시 확인한다. 약칭의 의도(`삼전`이 삼성전자를 뜻하는 상황 등)는 이 notebook의 **개발용 사용자 표현 가정**이다. OpenDART가 약칭을 공식적으로 보증한다는 뜻은 아니다.

- **A. 현재 방식**: 정규화 정확 일치가 있으면 그것만 반환하고, 없을 때만 fuzzy 후보를 반환한다.
- **B. 통합 후보 방식**: 정규화 정확 일치와 fuzzy 후보를 함께 반환한다.
- **C. 통합 + 소수 별칭 방식**: B에 개발용 소수 별칭 후보를 더한다.

공식 원천: [OpenDART 고유번호 API 개발가이드](https://opendart.fss.or.kr/guide/detail.do?apiGrpCd=DS001&apiId=2019018), [RapidFuzz process 문서](https://rapidfuzz.github.io/RapidFuzz/Usage/process.html).

In [ ]:
import io
from datetime import datetime
import os
import re
import sys
import time
import xml.etree.ElementTree as ET
import zipfile
from pathlib import Path
from statistics import median

from rapidfuzz import fuzz, process


In [ ]:
current_dir = Path.cwd().resolve()
if current_dir.name == 'v1':
    project_root = current_dir.parents[1]
elif current_dir.name == 'notebooks':
    project_root = current_dir.parent
else:
    project_root = current_dir

env_path = project_root / '.env'
if not env_path.exists():
    raise FileNotFoundError(f'{env_path} 파일을 먼저 만들어 주세요.')

for raw_line in env_path.read_text(encoding='utf-8').splitlines():
    line = raw_line.strip()
    if line and not line.startswith('#') and '=' in line:
        key, value = line.split('=', 1)
        os.environ.setdefault(key.strip(), value.strip().strip(chr(39) + chr(34)))

api_key = os.environ.get('OPENDART_API_KEY')
if not api_key:
    raise ValueError('.env에 OPENDART_API_KEY를 설정해 주세요.')


In [ ]:
sys.path.insert(0, str(project_root / 'src'))

from dart.client import CORP_CODE_URL, _get_bytes


## 2. OpenDART 기업코드 목록을 로드한다

후보는 별도 기업 목록이 아니라 OpenDART가 제공하는 기업코드 ZIP에서만 만든다. 따라서 후보의 법인명·`corp_code`·종목코드는 모두 같은 기준일의 OpenDART 데이터에 연결된다.

In [ ]:
def load_corp_records(api_key):
    zip_bytes = _get_bytes(CORP_CODE_URL, {'crtfc_key': api_key})
    zip_stream = io.BytesIO(zip_bytes)
    if not zipfile.is_zipfile(zip_stream):
        error_text = zip_bytes.decode('utf-8', errors='replace')
        raise RuntimeError(f'기업코드 응답이 ZIP이 아닙니다: {error_text}')
    with zipfile.ZipFile(zip_stream) as archive:
        xml_bytes = archive.read(archive.namelist()[0])

    root = ET.fromstring(xml_bytes)
    return [
        {child.tag: child.text or '' for child in item}
        for item in root.findall('list')
    ]

corp_records = load_corp_records(api_key)
retrieved_at = datetime.now().astimezone().isoformat(timespec='seconds')
print(f'OpenDART 기업코드 조회 시각: {retrieved_at}')
print(f'전체 법인 레코드 수: {len(corp_records):,}')


In [ ]:
def normalize_name(value):
    return re.sub(r'[^0-9A-Z가-힣]', '', value.upper())

def build_name_index(records):
    index = {}
    for record in records:
        for name in {record['corp_name'], record['corp_eng_name']}:
            key = normalize_name(name)
            if key:
                index.setdefault(key, {})[record['corp_code']] = record
    return {key: list(by_code.values()) for key, by_code in index.items()}

name_index = build_name_index(corp_records)
name_keys = list(name_index)
print(f'정규화 이름 키 수: {len(name_keys):,}')


## 3. 대표 기업과 개발용 사용자 표현을 정의한다

여기서 ‘대표’는 법적 대기업 전체를 뜻하지 않는다. 약칭·표기 차이 평가를 위한 대표적인 상장기업 표본이다. `기대 대상`은 정식 법인명과 종목코드로 OpenDART 목록에서 검증한다.

`엘지`, `현대`, `삼성`은 하나의 법인으로 단정하면 안 되는 **의도적 모호 입력**이다. 이 입력들은 특정 한 회사를 정답으로 두지 않고, 복수의 관련 후보가 유지되는지만 확인한다.

In [ ]:
TARGET_COMPANIES = {
    '삼성전자': {'official_name': '삼성전자', 'stock_code': '005930'},
    'SK하이닉스': {'official_name': 'SK하이닉스', 'stock_code': '000660'},
    'LG': {'official_name': 'LG', 'stock_code': '003550'},
    'LG전자': {'official_name': 'LG전자', 'stock_code': '066570'},
    'LG화학': {'official_name': 'LG화학', 'stock_code': '051910'},
    '현대자동차': {'official_name': '현대자동차', 'stock_code': '005380'},
    'NAVER': {'official_name': 'NAVER', 'stock_code': '035420'},
    '카카오': {'official_name': '카카오', 'stock_code': '035720'},
    'POSCO홀딩스': {'official_name': 'POSCO홀딩스', 'stock_code': '005490'},
    '삼성바이오로직스': {'official_name': '삼성바이오로직스', 'stock_code': '207940'},
    '셀트리온': {'official_name': '셀트리온', 'stock_code': '068270'},
    '기아': {'official_name': '기아', 'stock_code': '000270'},
}

target_records = {}
for target_key, target in TARGET_COMPANIES.items():
    matches = [
        record for record in corp_records
        if normalize_name(record['corp_name']) == normalize_name(target['official_name'])
        and record['stock_code'].strip() == target['stock_code']
    ]
    if len(matches) != 1:
        raise AssertionError(f'{target_key}: OpenDART 기준 법인 검증 실패 ({len(matches)}건)')
    target_records[target_key] = matches[0]

for key, record in target_records.items():
    print(f'{key}: {record["corp_name"]} / {record["corp_code"]} / {record["stock_code"]}')


In [ ]:
TEST_CASES = [
    {'질문 기업명': '삼성전자', '유형': '정식 명칭', '기대 대상': ['삼성전자']},
    {'질문 기업명': '삼성 전자', '유형': '띄어쓰기', '기대 대상': ['삼성전자']},
    {'질문 기업명': '삼전', '유형': '시장 약칭·동명', '기대 대상': ['삼성전자']},
    {'질문 기업명': '삼성전쟈', '유형': '한 글자 오타', '기대 대상': ['삼성전자']},
    {'질문 기업명': 'SK하이닉스', '유형': '정식 명칭', '기대 대상': ['SK하이닉스']},
    {'질문 기업명': 'sk 하이닉스', '유형': '영문 대소문자·띄어쓰기', '기대 대상': ['SK하이닉스']},
    {'질문 기업명': '하이닉스', '유형': '부분 표현', '기대 대상': ['SK하이닉스']},
    {'질문 기업명': '하닉', '유형': '시장 약칭', '기대 대상': ['SK하이닉스']},
    {'질문 기업명': 'LG전자', '유형': '정식 명칭', '기대 대상': ['LG전자']},
    {'질문 기업명': 'LG 전자', '유형': '띄어쓰기', '기대 대상': ['LG전자']},
    {'질문 기업명': '엘지전자', '유형': '한글 표기', '기대 대상': ['LG전자']},
    {'질문 기업명': '현대자동차', '유형': '정식 명칭', '기대 대상': ['현대자동차']},
    {'질문 기업명': '현대차', '유형': '시장 약칭', '기대 대상': ['현대자동차']},
    {'질문 기업명': '현차', '유형': '시장 약칭', '기대 대상': ['현대자동차']},
    {'질문 기업명': 'NAVER', '유형': '정식 영문 명칭', '기대 대상': ['NAVER']},
    {'질문 기업명': 'naver', '유형': '영문 소문자', '기대 대상': ['NAVER']},
    {'질문 기업명': '네이버', '유형': '한글 표기', '기대 대상': ['NAVER']},
    {'질문 기업명': '카카오', '유형': '정식 명칭', '기대 대상': ['카카오']},
    {'질문 기업명': 'POSCO홀딩스', '유형': '정식 영문 혼합 명칭', '기대 대상': ['POSCO홀딩스']},
    {'질문 기업명': '포스코홀딩스', '유형': '한글 표기', '기대 대상': ['POSCO홀딩스']},
    {'질문 기업명': '삼성바이오로직스', '유형': '정식 명칭', '기대 대상': ['삼성바이오로직스']},
    {'질문 기업명': '삼바', '유형': '시장 약칭', '기대 대상': ['삼성바이오로직스']},
    {'질문 기업명': '셀트리온', '유형': '정식 명칭', '기대 대상': ['셀트리온']},
    {'질문 기업명': '셀트', '유형': '시장 약칭', '기대 대상': ['셀트리온']},
    {'질문 기업명': '기아', '유형': '정식 명칭', '기대 대상': ['기아']},
    {'질문 기업명': '기아차', '유형': '시장 약칭', '기대 대상': ['기아']},
    {'질문 기업명': '엘지', '유형': '의도적 모호 표현', '기대 대상': ['LG', 'LG전자', 'LG화학']},
    {'질문 기업명': '현대', '유형': '의도적 모호 표현', '기대 대상': ['현대자동차', '기아']},
    {'질문 기업명': '삼성', '유형': '의도적 모호 표현', '기대 대상': ['삼성전자', '삼성바이오로직스']},
]

print(f'전체 평가 입력: {len(TEST_CASES)}개')
print(f'하나의 기대 대상: {sum(len(case["기대 대상"]) == 1 for case in TEST_CASES)}개')
print(f'복수 후보가 정상인 입력: {sum(len(case["기대 대상"]) > 1 for case in TEST_CASES)}개')


## 4. 세 후보 생성 방식을 정의한다

별칭은 아직 최종 운영 사전이 아니다. C는 **작은 별칭 사전을 추가하면 후보 누락이 실제로 줄어드는지** 보기 위한 진단용 비교안이다. 별칭으로 추가된 후보는 fuzzy 점수와 같은 숫자로 비교하지 않고, `alias` 출처로 따로 표시한다.

In [ ]:
FUZZY_KEY_LIMIT = 50
FUZZY_RESULT_LIMIT = 5

# 개발용 가설이다. 운영 사전으로 채택하지 않았으며, 이번 비교 뒤 독립 holdout으로 다시 검증한다.
CURATED_ALIAS_TARGETS = {
    normalize_name('삼전'): ['삼성전자'],
    normalize_name('하이닉스'): ['SK하이닉스'],
    normalize_name('하닉'): ['SK하이닉스'],
    normalize_name('엘지전자'): ['LG전자'],
    normalize_name('현대차'): ['현대자동차'],
    normalize_name('현차'): ['현대자동차'],
    normalize_name('네이버'): ['NAVER'],
    normalize_name('삼바'): ['삼성바이오로직스'],
    normalize_name('셀트'): ['셀트리온'],
    normalize_name('기아차'): ['기아'],
    normalize_name('엘지'): ['LG', 'LG전자', 'LG화학'],
    normalize_name('현대'): ['현대자동차', '기아'],
    normalize_name('삼성'): ['삼성전자', '삼성바이오로직스'],
}

def add_candidate(candidates, record, source, fuzzy_score=None):
    current = candidates.setdefault(
        record['corp_code'],
        {'record': record, 'sources': set(), 'fuzzy_score': None},
    )
    current['sources'].add(source)
    if fuzzy_score is not None:
        current['fuzzy_score'] = max(current['fuzzy_score'] or 0, fuzzy_score)

def fuzzy_candidate_records(query):
    scored_keys = process.extract(
        normalize_name(query), name_keys, scorer=fuzz.ratio, limit=FUZZY_KEY_LIMIT
    )
    best_by_code = {}
    for key, score, _ in scored_keys:
        for record in name_index[key]:
            current = best_by_code.get(record['corp_code'])
            if current is None or score > current[1]:
                best_by_code[record['corp_code']] = (record, score)
    return sorted(
        best_by_code.values(),
        key=lambda item: (item[1], bool(item[0]['stock_code'].strip()), item[0]['modify_date']),
        reverse=True,
    )[:FUZZY_RESULT_LIMIT]

def exact_candidates(query):
    return name_index.get(normalize_name(query), [])

def current_candidates(query):
    exact_records = exact_candidates(query)
    candidates = {}
    if exact_records:
        for record in exact_records:
            add_candidate(candidates, record, 'exact')
        return candidates
    for record, score in fuzzy_candidate_records(query):
        add_candidate(candidates, record, 'fuzzy', score)
    return candidates

def union_candidates(query):
    candidates = {}
    for record in exact_candidates(query):
        add_candidate(candidates, record, 'exact')
    for record, score in fuzzy_candidate_records(query):
        add_candidate(candidates, record, 'fuzzy', score)
    return candidates

def union_with_alias_candidates(query):
    candidates = union_candidates(query)
    for target_key in CURATED_ALIAS_TARGETS.get(normalize_name(query), []):
        add_candidate(candidates, target_records[target_key], 'alias')
    return candidates


## 5. 정량 비교: 정답 후보 포함 여부와 후보 수

후보 출처마다 점수의 의미가 다르므로 이번에는 순위 기반 `Recall@K`를 쓰지 않는다. 대신 아래를 비교한다.

- **Candidate Recall**: 하나의 기대 대상이 후보 집합에 포함되었는가
- **Ambiguous Candidate Recall**: 모호 입력에서 미리 정한 관련 후보가 얼마나 유지되었는가
- **Candidate Set Size**: 한 입력에서 사용자가 볼 후보 수가 과도하게 늘어나는가
- **p50/p95 지연시간**: 후보 통합·별칭 추가의 시간 비용


In [ ]:
WARMUP_ITERATIONS = 2
MEASURE_ITERATIONS = 10

CANDIDATE_METHODS = {
    'A. 현재 exact-first': current_candidates,
    'B. exact + fuzzy 통합': union_candidates,
    'C. B + 개발용 별칭': union_with_alias_candidates,
}

def candidate_view(candidate):
    record = candidate['record']
    return {
        '법인명': record['corp_name'],
        'corp_code': record['corp_code'],
        '종목코드': record['stock_code'].strip() or None,
        '출처': '+'.join(sorted(candidate['sources'])),
        '문자열 유사 점수': round(candidate['fuzzy_score'], 1) if candidate['fuzzy_score'] is not None else None,
    }

def evaluate_method(method_name, finder):
    rows = []
    latencies = []
    for case in TEST_CASES:
        for _ in range(WARMUP_ITERATIONS):
            finder(case['질문 기업명'])

        elapsed = []
        for _ in range(MEASURE_ITERATIONS):
            started_at = time.perf_counter()
            candidates = finder(case['질문 기업명'])
            elapsed.append((time.perf_counter() - started_at) * 1_000)

        expected_codes = {target_records[key]['corp_code'] for key in case['기대 대상']}
        candidate_codes = set(candidates)
        views = sorted(
            (candidate_view(candidate) for candidate in candidates.values()),
            key=lambda row: (row['출처'], -(row['문자열 유사 점수'] or 0), row['법인명']),
        )
        rows.append({
            **case,
            '방법': method_name,
            '정답 후보 포함 수': len(expected_codes & candidate_codes),
            '후보 수': len(candidates),
            '후보': views,
        })
        latencies.extend(elapsed)

    single_rows = [row for row in rows if len(row['기대 대상']) == 1]
    ambiguous_rows = [row for row in rows if len(row['기대 대상']) > 1]
    summary = {
        '방법': method_name,
        'Candidate Recall': sum(row['정답 후보 포함 수'] == 1 for row in single_rows) / len(single_rows),
        'Ambiguous Candidate Recall': (
            sum(row['정답 후보 포함 수'] for row in ambiguous_rows)
            / sum(len(row['기대 대상']) for row in ambiguous_rows)
        ),
        '평균 후보 수': sum(row['후보 수'] for row in rows) / len(rows),
        '후보 수 중앙값': median(row['후보 수'] for row in rows),
        'p50 지연시간(ms)': median(latencies),
        'p95 지연시간(ms)': sorted(latencies)[int(len(latencies) * 0.95) - 1],
    }
    return rows, summary

all_rows = {}
summaries = []
for method_name, finder in CANDIDATE_METHODS.items():
    rows, summary = evaluate_method(method_name, finder)
    all_rows[method_name] = rows
    summaries.append(summary)

for summary in summaries:
    print(summary)


## 6. 핵심 입력의 후보를 직접 확인한다

특히 `삼전`은 OpenDART에 같은 이름의 법인이 존재하므로, 삼성전자가 후보에 추가되더라도 자동 선택하면 안 된다. `엘지`·`현대`·`삼성`도 여러 후보가 유지되는 것이 정상이다.

In [ ]:
FOCUS_INPUTS = ['삼전', '하닉', '엘지전자', '엘지', '현대', '삼성', '삼성전쟈']

for query in FOCUS_INPUTS:
    print(f'\n=== {query} ===')
    for method_name, rows in all_rows.items():
        row = next(row for row in rows if row['질문 기업명'] == query)
        print(f'{method_name}: 정답 후보 포함 수={row["정답 후보 포함 수"]}, 후보 수={row["후보 수"]}')
        for candidate in row['후보']:
            print(' ', candidate)


## 7. 실행 결과 해석 기준

- B가 A보다 Candidate Recall을 높이면, 정확 일치가 fuzzy 후보를 가리는 현재 구조의 한계를 확인한 것이다.
- C가 B보다 약칭·한글 표기에서 Candidate Recall을 높이면, 개발용 소수 별칭이 해당 표현을 기계적으로 복구함을 보인 것이다. 이번 별칭과 평가 입력은 같은 개발 가정에서 만들었으므로, 미지 표현 일반화 성능이나 운영 정책 채택 근거는 아니다.
- C의 후보 수가 지나치게 늘거나, 모호 입력을 하나의 후보로 축소하면 미채택 또는 수정 대상이다.
- 이번 결과를 본 뒤, 실제로 지원할 별칭 범위와 독립 holdout 평가셋을 정한다. 그 후에야 후보 생성·정렬·사용자 확인 정책을 함께 확정한다.

이 notebook은 최종 pipeline이나 `src/` 코드를 변경하지 않는다.